# Lab 10: Production Patterns

This lab covers **production concerns** for running Quant-Sports reliably:

- Scheduling strategies (cron, Prefect, Kubernetes CronJob)
- Error handling patterns (retry, circuit breaker)
- Data quality checks
- Alerting on poller failures
- Retention policy tuning
- Backup strategies
- Monitoring with Prometheus
- Health endpoints and graceful shutdown
- Logging best practices
- Configuration management

By the end, you will have a production-ready checklist for deploying Quant-Sports at scale.

---

## Prerequisites

- **Labs 01-09 completed** — you understand strategies, backtesting, and custom strategy implementation
- **The Quant-Sports poller** — basic familiarity with PollerConfig and the poller CLI
- **Docker and docker-compose** — for deployment (optional)

### Production Architecture Overview

```
+------------+     +-------------+     +------------+
|  Odds API  |     | ESPN Injuries|    | Other APIs |
|  (Poller)  |     |  (Poller)    |    |  (Future)  |
+-----+------+     +------+-------+    +-----+------+
      |                   |                  |
      +---------+---------+------------------+
                |
                v
       +----------------+
       |  TimescaleDB   |  <- Data storage
       +-------+--------+
               |
    +----------+----------+----------+
    |          |          |          |
    v          v          v          v
+--------+ +--------+ +--------+ +--------+
|Strategy| |Alerting| |Metrics | |Grafana |
| Engine | | System | | Export | |Dashboard|
+--------+ +--------+ +--------+ +--------+
```

---

## Section 1: Setup - Imports and Configuration

We import the poller configuration, database utilities, and standard libraries for error handling, scheduling, and monitoring.

In [ ]:
import asyncio
import json
import logging
import signal
import time
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Optional

import numpy as np
import pandas as pd

# Quant-Sports imports
from quantitative_sports.infra.poller.config import (
    PollerConfig,
    get_active_sports,
    is_odds_api_enabled,
    is_espn_enabled,
)
from quantitative_sports.infra.poller.tasks import TaskResult, run_poll_cycle, run_health_check

# Database
from quantitative_sports.infra.db.connection import DBConfig, DatabasePool, get_pool, health_check

# Enable nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("lab10")

print("Imports loaded successfully.")
print(f"PollerConfig defaults: scheduler={PollerConfig().scheduler}, "
      f"sports={PollerConfig().active_sports}")

---

## Section 2: Scheduling Strategies

Quant-Sports supports multiple scheduling backends. Each has trade-offs:

| Scheduler | Pros | Cons |
|---|---|---|
| **Prefect** | Dashboard, retries, observability | Requires server |
| **cron** | Simple, no dependencies | No dashboard, manual error handling |
| **Kubernetes CronJob** | Cloud-native, auto-restart | Complex setup, K8s required |

The `PollerConfig.scheduler` field selects the backend:
- `"prefect"` uses `serve_flows()` with Prefect deployments
- `"cron"` uses `run_cron_loop()` in a plain asyncio loop

In [ ]:
default_config = PollerConfig()
print("=== Default PollerConfig ===")
print(f"  Scheduler: {default_config.scheduler}")
print(f"  Active sports: {get_active_sports(default_config)}")
print(f"  Odds API enabled: {is_odds_api_enabled(default_config)}")
print(f"  ESPN enabled: {is_espn_enabled(default_config)}")
print(f"  Max retries: {default_config.max_retries}")
print(f"  Retry delay: {default_config.retry_delay}s")
print(f"  Log level: {default_config.log_level}")
print(f"  DB host: {default_config.db.host}")
print(f"  DB port: {default_config.db.port}")

print("\n=== Scheduling Backends ===")
print()
print("1. Prefect (recommended for production):")
print("   QUANT_SPORTS_POLLER_SCHEDULER=prefect")
print("   QUANT_SPORTS_POLLER_PREFECT_API_URL=http://localhost:4200/api")
print()
print("2. cron (simplest, good for development):")
print("   QUANT_SPORTS_POLLER_SCHEDULER=cron")
print("   # Runs an asyncio loop with sleep intervals")
print()
print("3. Kubernetes CronJob (cloud deployment):")
print("   # In your manifest:")
print("   #   kind: CronJob")
print("   #   schedule: every 5 minutes")
print("   #   command: [quant-sports-poller, once]")

---

## Section 3: Error Handling Patterns

Production systems must handle failures gracefully. Quant-Sports poller already implements error handling in tasks.py - each task catches all exceptions and returns a TaskResult with status and error details.

Key patterns:
1. **Try/except per task** - one failing poller never kills the others
2. **Retry with backoff** - exponential backoff for transient failures
3. **Circuit breaker** - stop calling a failing service after N failures
4. **Health tracking** - `poller_health` table tracks consecutive failures

In [ ]:
import asyncio
from functools import wraps

# 1. Simple retry with backoff
async def retry_with_backoff(func, max_retries=3, base_delay=1.0, max_delay=60.0):
    """Call an async function with exponential backoff retry."""
    last_exception = None
    for attempt in range(max_retries + 1):
        try:
            return await func()
        except Exception as e:
            last_exception = e
            if attempt < max_retries:
                delay = min(base_delay * (2 ** attempt), max_delay)
                logger.warning(
                    "retry_with_backoff: attempt %d/%d failed (%s), retrying in %.1fs",
                    attempt + 1, max_retries + 1, type(e).__name__, delay
                )
                await asyncio.sleep(delay)
            else:
                logger.error("retry_with_backoff: all %d attempts failed", max_retries + 1)
    raise last_exception


# 2. Circuit breaker
class CircuitBreaker:
    """Simple circuit breaker that opens after consecutive failures."""
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

    def __init__(self, failure_threshold=3, recovery_timeout=30.0, name="default"):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.name = name
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0.0

    def record_success(self):
        """Record a successful call - reset the breaker."""
        self.failure_count = 0
        self.state = self.CLOSED

    def record_failure(self):
        """Record a failed call - may trip the breaker."""
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = self.OPEN
            logger.warning("CircuitBreaker(%s): OPEN after %d failures", self.name, self.failure_count)

    def can_execute(self):
        """Check if a request can proceed."""
        if self.state == self.CLOSED:
            return True
        if self.state == self.OPEN:
            elapsed = time.time() - self.last_failure_time
            if elapsed >= self.recovery_timeout:
                self.state = self.HALF_OPEN
                logger.info("CircuitBreaker(%s): HALF_OPEN after %.1fs", self.name, elapsed)
                return True
            return False
        return True  # HALF_OPEN allows one request

    @property
    def status(self):
        return {"name": self.name, "state": self.state,
                "failure_count": self.failure_count,
                "failure_threshold": self.failure_threshold}


# Demo
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5.0, name="odds_api")
print("Circuit breaker initialized:")
print(f"  State: {breaker.state}")
print(f"  Failure threshold: {breaker.failure_threshold}")

for i in range(4):
    breaker.record_failure()
    print(f"  After failure {i+1}: state={breaker.state}, failures={breaker.failure_count}")
print(f"  Can execute: {breaker.can_execute()}")

---

## Section 4: Data Quality Checks

Before trusting any data, you should verify:

1. **Schema** - are the expected columns present with correct types?
2. **Freshness** - is the data recent enough to be useful?
3. **Completeness** - are there null values or missing rows?
4. **Range** - are values within expected bounds?

Here we implement a lightweight DataQualityChecker that can run as part of the poller pipeline.

In [ ]:
@dataclass
class QualityReport:
    """Result of a data quality check."""
    check_name: str
    passed: bool
    details: str
    severity: str = "info"


class DataQualityChecker:
    """Validate incoming odds data against quality rules."""
    EXPECTED_COLUMNS = {"event_id", "book", "market", "selection", "price", "line", "ts"}

    def __init__(self, max_age_hours=1.0):
        self.max_age_hours = max_age_hours

    def check_schema(self, df):
        """Verify the DataFrame has expected columns."""
        missing = self.EXPECTED_COLUMNS - set(df.columns)
        if missing:
            return QualityReport("schema", False, f"Missing columns: {missing}", "error")
        return QualityReport("schema", True, f"All {len(self.EXPECTED_COLUMNS)} expected columns present")

    def check_freshness(self, df, ts_column="ts"):
        """Verify data is not too old."""
        if ts_column not in df.columns:
            return QualityReport("freshness", False, f"Column {ts_column} not found", "error")
        try:
            latest = pd.to_datetime(df[ts_column].max())
            age = pd.Timestamp.now(tz="UTC") - latest.tz_localize("UTC")
            if age > timedelta(hours=self.max_age_hours):
                hours = age.total_seconds() / 3600
                return QualityReport("freshness", False, f"Data is {hours:.1f}h old (max: {self.max_age_hours}h)", "warning")
            minutes = age.total_seconds() / 60
            return QualityReport("freshness", True, f"Data is {minutes:.0f} minutes old")
        except Exception as e:
            return QualityReport("freshness", False, f"Parse error: {e}", "warning")

    def check_completeness(self, df):
        """Check for null values in required columns."""
        required = {"event_id", "book", "market", "selection", "price"}
        null_counts = {}
        for col in required:
            if col in df.columns:
                nulls = df[col].isna().sum()
                if nulls > 0:
                    null_counts[col] = int(nulls)
        if null_counts:
            return QualityReport("completeness", False, f"Null values: {null_counts}", "warning")
        return QualityReport("completeness", True, f"No nulls in {len(required)} required columns")

    def check_row_count(self, df, min_rows=1):
        """Verify sufficient data volume."""
        if len(df) < min_rows:
            return QualityReport("row_count", False, f"Only {len(df)} rows (minimum: {min_rows})", "warning")
        return QualityReport("row_count", True, f"{len(df)} rows (minimum: {min_rows})")

    def run_all(self, df):
        return [self.check_schema(df), self.check_freshness(df),
                self.check_completeness(df), self.check_row_count(df)]


# Demo
checker = DataQualityChecker(max_age_hours=1.0)
sample_data = pd.DataFrame({
    "event_id": ["KC vs BUF", "KC vs BUF", "SF vs PHI"],
    "book": ["fanduel", "draftkings", "pinnacle"],
    "market": ["h2h", "h2h", "spreads"],
    "selection": ["KC", "BUF", "SF -3.5"],
    "price": [-110, -105, -110],
    "line": [None, None, -3.5],
    "ts": [datetime.now(timezone.utc).isoformat()] * 3,
})

print("Data Quality Check Results:")
print("=" * 60)
for report in checker.run_all(sample_data):
    status = "PASS" if report.passed else "FAIL"
    severity_tag = f"[{report.severity}]" if not report.passed else ""
    print(f"  {status} {report.check_name}: {report.details} {severity_tag}")

---

## Section 5: Alerting on Poller Failures

The `poller_health` table tracks the status of each poller. A poller is **down** after 3 consecutive failures. We can query this table to detect problems and send alerts.

The poller tracks:
- `status`: "healthy", "degraded", or "down"
- `consecutive_failures`: how many times in a row it failed
- `last_checked_at`: timestamp of last health check

In [ ]:
@dataclass
class Alert:
    """An alert from the monitoring system."""
    level: str
    source: str
    message: str
    timestamp: str


class AlertManager:
    """Check poller health and generate alerts."""
    FAILURE_THRESHOLD = 3

    def __init__(self, db_pool=None):
        self.pool = db_pool
        self.alerts = []

    async def check_health(self):
        """Query poller_health and generate alerts."""
        alerts = []
        if self.pool is None:
            alerts.append(Alert("info", "odds_api_nfl", "Poller is healthy",
                datetime.now(timezone.utc).isoformat()))
            alerts.append(Alert("warning", "espn_injuries_nfl", "Consecutive failures: 2 - degraded",
                datetime.now(timezone.utc).isoformat()))
            return alerts

        rows = await self.pool.fetch(
            "SELECT poller_name, status, consecutive_failures, last_checked_at FROM poller_health")
        for row in rows:
            failures = row["consecutive_failures"]
            status = row["status"]
            name = row["poller_name"]
            if status == "down" or failures >= self.FAILURE_THRESHOLD:
                level = "critical"
                msg = f"DOWN - {failures} consecutive failures"
            elif status == "degraded" or failures >= 1:
                level = "warning"
                msg = f"DEGRADED - {failures} consecutive failures"
            else:
                level = "info"
                msg = "Healthy - no issues"
            alerts.append(Alert(level, name, msg, str(row.get("last_checked_at", ""))))
        self.alerts.extend(alerts)
        return alerts

    def format_alerts(self, alerts):
        """Format alerts for display."""
        order = {"critical": 0, "warning": 1, "info": 2}
        icons = {"critical": "RED", "warning": "YELLOW", "info": "GREEN"}
        lines = []
        for a in sorted(alerts, key=lambda x: order.get(x.level, 99)):
            lines.append(f"{icons.get(a.level, "?")} [{a.level.upper()}] {a.source}: {a.message}")
        return "\n".join(lines)


# Demo
alert_mgr = AlertManager()
alerts = asyncio.run(alert_mgr.check_health())
print("Poller Health Alerts:")
print("=" * 60)
print(alert_mgr.format_alerts(alerts))

---

## Section 6: Retention Policy Tuning

TimescaleDB hypertables support automatic data retention via `drop_chunks`. Here are recommended retention periods for each table:

| Table | Recommended Retention | Reason |
|---|---|---|
| `odds_ticks` | 90 days | Odds data ages quickly; 90 days for backtesting |
| `injuries` | 365 days | Injury history has seasonal patterns |
| `poller_runs` | 30 days | Operational data, short retention |
| `poller_health` | 30 days | Operational data, short retention |
| `bets` | Indefinite | Betting history is your P&L record |

In [ ]:
retention_policies = [
    {"table": "odds_ticks", "retention": "90 days",
     "reason": "Odds data ages quickly; 90 days for backtesting"},
    {"table": "injuries", "retention": "365 days",
     "reason": "Injury history has seasonal patterns; keep 1 year"},
    {"table": "poller_runs", "retention": "30 days",
     "reason": "Operational data; short retention is sufficient"},
    {"table": "poller_health", "retention": "30 days",
     "reason": "Operational data; short retention is sufficient"},
]

print("Recommended Retention Policies:")
print("=" * 70)
for policy in retention_policies:
    print(f"  Table: {policy['table']}")
    print(f"  Retention: {policy['retention']}")
    print(f"  Reason: {policy['reason']}")
    print()

print("=" * 70)
print("Retention SQL examples:")
print("  SELECT drop_chunks('odds_ticks', INTERVAL '90 days');")
print("  SELECT drop_chunks('injuries', INTERVAL '365 days');")
print("  SELECT drop_chunks('poller_runs', INTERVAL '30 days');")
print("  SELECT drop_chunks('poller_health', INTERVAL '30 days');")

---

## Section 7: Backup Strategies

Data loss is unacceptable in a betting system. Here are backup strategies:

1. **pg_dump** - Full logical backup (daily)
2. **TimescaleDB continuous aggregates** - Summary tables that auto-refresh
3. **WAL archiving** - Point-in-time recovery (PITR)
4. **Replication** - Hot standby for read scaling and failover

In [ ]:
print("Backup Strategy: pg_dump")
print("=" * 60)
print("""
#!/bin/bash
# Daily full backup of the Quant-Sports database
set -euo pipefail
DB_HOST=${QUANT_SPORTS_DB_HOST:-localhost}
DB_PORT=${QUANT_SPORTS_DB_PORT:-5434}
DB_NAME=${QUANT_SPORTS_DB_NAME:-quantitative_sports}
DB_USER=${QUANT_SPORTS_DB_USER:-quantitative_sports}
BACKUP_DIR="/opt/quantitative_sports/backups"
TIMESTAMP=$(date +%Y%m%d_%H%M%S)
BACKUP_FILE="${BACKUP_DIR}/quantitative_sports_${TIMESTAMP}.sql.gz"
mkdir -p "${BACKUP_DIR}"
echo "[$(date)] Starting backup..."
pg_dump -h "${DB_HOST}" -p "${DB_PORT}" -U "${DB_USER}" --format=custom --compress=9 "${DB_NAME}" > "${BACKUP_FILE}"
echo "[$(date)] Backup complete: ${BACKUP_FILE}"
find "${BACKUP_DIR}" -name "*.sql.gz" -mtime +30 -delete
""")

print("\nContinuous Aggregate (for summary stats):")
print("""
CREATE MATERIALIZED VIEW odds_hourly
WITH (timescaledb.continuous) AS
SELECT time_bucket('1 hour', ts) AS bucket,
       event_id, book, market,
       AVG(price) AS avg_price,
       MIN(price) AS min_price,
       MAX(price) AS max_price,
       COUNT(*) AS tick_count
FROM odds_ticks
GROUP BY bucket, event_id, book, market;

-- Refresh policy (auto-refresh every hour)
SELECT add_continuous_aggregate_policy('odds_hourly',
    start_offset => INTERVAL '3 hours',
    end_offset => INTERVAL '1 hour',
    schedule_interval => INTERVAL '1 hour');
""")

---

## Section 8: Monitoring with Prometheus

Prometheus is the standard for application metrics. We expose metrics using `prometheus_client` that track poller health, strategy performance, and database connectivity.

In [ ]:
PROMETHEUS_AVAILABLE = False
try:
    from prometheus_client import Counter, Gauge, Histogram, start_http_server
    PROMETHEUS_AVAILABLE = True
    print("prometheus_client is available")
except ImportError:
    print("prometheus_client not installed. Install with: pip install prometheus_client")

if PROMETHEUS_AVAILABLE:
    POLLER_CYCLES_TOTAL = Counter(
        "quantitative_sports_poller_cycles_total",
        "Total number of poller cycles",
        ["poller_name", "status"])
    POLLER_CYCLE_DURATION = Histogram(
        "quantitative_sports_poller_cycle_duration_seconds",
        "Duration of poller cycles in seconds",
        ["poller_name"],
        buckets=[0.5, 1.0, 2.5, 5.0, 10.0, 30.0, 60.0])
    ODDS_TICKS_INGESTED = Counter(
        "quantitative_sports_odds_ticks_ingested_total",
        "Total odds ticks ingested",
        ["sport", "book"])
    STRATEGY_BETS_TOTAL = Counter(
        "quantitative_sports_strategy_bets_total",
        "Total bets placed by strategy",
        ["strategy", "side", "result"])
    STRATEGY_BANKROLL = Gauge(
        "quantitative_sports_strategy_bankroll_dollars",
        "Current bankroll for each strategy",
        ["strategy"])
    DB_CONNECTION_POOL_SIZE = Gauge(
        "quantitative_sports_db_pool_size",
        "Database connection pool size")
    CONSECUTIVE_FAILURES = Gauge(
        "quantitative_sports_poller_consecutive_failures",
        "Consecutive failures per poller",
        ["poller_name"])
    print("Defined metrics:")
    print("  - quantitative_sports_poller_cycles_total")
    print("  - quantitative_sports_poller_cycle_duration_seconds")
    print("  - quantitative_sports_odds_ticks_ingested_total")
    print("  - quantitative_sports_strategy_bets_total")
    print("  - quantitative_sports_strategy_bankroll_dollars")
    print("  - quantitative_sports_db_pool_size")
    print("  - quantitative_sports_poller_consecutive_failures")
else:
    print("Skipping Prometheus metrics (prometheus_client not installed)")

---

## Section 9: Grafana Dashboard

Grafana connects to Prometheus for metrics and TimescaleDB for data queries. Here are example PromQL queries and SQL panels for your dashboard.

In [ ]:
grafana_panels = {
    "Poller Health": {
        "type": "stat",
        "promql": "quantitative_sports_poller_consecutive_failures",
        "description": "Consecutive failures per poller",
    },
    "Cycle Duration": {
        "type": "time_series",
        "promql": "histogram_quantile(0.95, rate(quantitative_sports_poller_cycle_duration_seconds_bucket[5m]))",
        "description": "95th percentile cycle duration",
    },
    "Odds Ingestion Rate": {
        "type": "time_series",
        "promql": "rate(quantitative_sports_odds_ticks_ingested_total[5m])",
        "description": "Ticks ingested per second",
    },
    "Strategy P/L": {
        "type": "time_series",
        "promql": "quantitative_sports_strategy_bankroll_dollars",
        "description": "Bankroll over time by strategy",
    },
}

print("Grafana Dashboard Panels:")
print("=" * 60)
for name, config in grafana_panels.items():
    print(f"  {name} ({config['type']})")
    print(f"  Description: {config['description']}")
    print(f"  PromQL: {config['promql']}")
    print()

---

## Section 10: Health Endpoints

A production system should expose three endpoints:

- `/health` - Is the process alive? (for load balancers)
- `/ready` - Is the system ready to serve? (checks DB, API keys)
- `/metrics` - Prometheus metrics (for scraping)

The Quant-Sports poller uses `run_health_check()` from tasks.py for database connectivity checks.

In [ ]:
@dataclass
class HealthStatus:
    """Health check result."""
    status: str
    checks: dict
    timestamp: str

    @property
    def is_healthy(self):
        return self.status == "healthy"

    @property
    def is_ready(self):
        return self.status != "down"


async def check_system_health(config):
    """Run all health checks and return aggregate status."""
    checks = {}

    # 1. Database connectivity
    try:
        pool = await get_pool(config.db)
        db_health = await health_check(pool)
        latency = db_health.get("latency_ms", "unknown")
        checks["database"] = (db_health["status"] == "healthy", f"Latency: {latency}ms")
        await pool.close()
    except Exception as e:
        checks["database"] = (False, f"Error: {e}")

    # 2. API key availability
    api_ok = is_odds_api_enabled(config)
    checks["odds_api"] = (api_ok, "API key configured" if api_ok else "No API key")
    checks["espn"] = (is_espn_enabled(config), "ESPN source enabled")

    # 3. Active sports configuration
    sports = get_active_sports(config)
    checks["sports_config"] = (len(sports) > 0, f"Active sports: {sports}")

    # 4. Scheduler
    checks["scheduler"] = (config.scheduler in ("prefect", "cron"), f"Scheduler: {config.scheduler}")

    # Aggregate status
    all_ok = all(ok for ok, _ in checks.values())
    any_down = any(not ok for ok, _ in checks.values())
    if all_ok:
        status = "healthy"
    elif any_down:
        status = "down"
    else:
        status = "degraded"

    return HealthStatus(status=status, checks=checks,
        timestamp=datetime.now(timezone.utc).isoformat())


# Demo
config = PollerConfig()
print("System Health Check (synthetic):")
print("=" * 60)
print(f"  Scheduler: {config.scheduler}")
print(f"  Active sports: {get_active_sports(config)}")
print(f"  Odds API enabled: {is_odds_api_enabled(config)}")
print(f"  ESPN enabled: {is_espn_enabled(config)}")
print(f"  DB host: {config.db.host}:{config.db.port}")
print(f"  Max retries: {config.max_retries}")

---

## Section 11: Graceful Shutdown

Production services must handle SIGTERM gracefully:
1. Stop accepting new work
2. Complete in-flight tasks
3. Close database connections
4. Flush metrics
5. Exit cleanly

The poller `run_flows_forever()` in flows.py already handles SIGINT/SIGTERM via asyncio.Event.

In [ ]:
class GracefulShutdown:
    """Handle SIGTERM/SIGINT for graceful shutdown."""
    def __init__(self):
        self.shutdown_event = asyncio.Event()
        self.in_flight_tasks = set()

    def request_shutdown(self, signum=None, frame=None):
        """Signal handler for graceful shutdown."""
        signal_name = signal.Signals(signum).name if signum else "UNKNOWN"
        logger.info("GracefulShutdown: received %s, initiating shutdown", signal_name)
        self.shutdown_event.set()

    def register_handlers(self):
        """Register signal handlers for SIGINT and SIGTERM."""
        loop = asyncio.get_event_loop()
        for sig in (signal.SIGINT, signal.SIGTERM):
            loop.add_signal_handler(sig, self.request_shutdown, sig)
        logger.info("GracefulShutdown: signal handlers registered")

    @property
    def should_shutdown(self):
        """Check if shutdown has been requested."""
        return self.shutdown_event.is_set()

    async def wait_for_shutdown(self, timeout=30.0):
        """Wait for shutdown signal with timeout."""
        try:
            await asyncio.wait_for(self.shutdown_event.wait(), timeout=timeout)
        except asyncio.TimeoutError:
            logger.warning("GracefulShutdown: timeout waiting for shutdown")

    async def wait_for_in_flight(self, timeout=30.0):
        """Wait for in-flight tasks to complete."""
        if not self.in_flight_tasks:
            return
        logger.info("GracefulShutdown: waiting for %d in-flight tasks", len(self.in_flight_tasks))
        done, pending = await asyncio.wait(self.in_flight_tasks, timeout=timeout)
        if pending:
            logger.warning("GracefulShutdown: %d tasks did not complete", len(pending))
            for task in pending:
                task.cancel()


# Demo
shutdown = GracefulShutdown()
print("GracefulShutdown pattern initialized")
print(f"  Should shutdown: {shutdown.should_shutdown}")
print("  Signal handler: will set event on SIGINT/SIGTERM")

---

## Section 12: Logging Best Practices

Good logging is essential for debugging production issues:

1. **Structured logging** - Use JSON for machine-parseable logs
2. **Log levels** - DEBUG, INFO, WARNING, ERROR, CRITICAL
3. **Correlation IDs** - Track a request across services
4. **Context** - Include poller_name, sport, run_id in every log line

In [ ]:
import json
import uuid
from logging import Formatter, StreamHandler


class JSONFormatter(Formatter):
    """Format log records as JSON for structured logging."""
    def format(self, record):
        log_entry = {
            "timestamp": self.formatTime(record, self.datefmt),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "correlation_id": getattr(record, "correlation_id", None),
            "poller_name": getattr(record, "poller_name", None),
            "sport": getattr(record, "sport", None),
            "run_id": getattr(record, "run_id", None),
        }
        log_entry = {k: v for k, v in log_entry.items() if v is not None}
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry)


def setup_structured_logging(level="INFO"):
    """Configure structured JSON logging for the poller."""
    root_logger = logging.getLogger("quantitative_sports")
    root_logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    root_logger.handlers.clear()
    handler = StreamHandler()
    handler.setFormatter(JSONFormatter())
    root_logger.addHandler(handler)
    return root_logger


# Demo
struct_logger = setup_structured_logging("INFO")
record = logging.LogRecord(
    name="quantitative_sports.poller.odds_api_nfl",
    level=logging.INFO,
    pathname=__file__,
    lineno=1,
    msg="Poll cycle completed",
    args=(),
    exc_info=None,
)
record.correlation_id = str(uuid.uuid4())
record.poller_name = "odds_api_nfl"
record.sport = "nfl"
record.run_id = str(uuid.uuid4())

formatter = JSONFormatter()
print("Structured Log Entry:")
print(formatter.format(record))

print("\n--- Logging Levels Guide ---")
for level_name, level_num in [("DEBUG", 10), ("INFO", 20), ("WARNING", 30), ("ERROR", 40), ("CRITICAL", 50)]:
    desc = {10: "Detailed diagnostic info (development only)",
            20: "General operational messages (poller cycles, strategy decisions)",
            30: "Unexpected but recoverable (retry attempts, degraded performance)",
            40: "Failures that need attention (API errors, DB connection lost)",
            50: "System-wide failures (cannot start, data corruption)"}[level_num]
    print(f"  {level_name} ({level_num}): {desc}")

---

## Section 13: Configuration Management

Quant-Sports uses `pydantic-settings` for configuration. Best practices:

1. **Environment variables** - all config via `QUANT_SPORTS_POLLER_*` env vars
2. **Secrets** - never commit API keys; use env vars or secret managers
3. **Validation** - Pydantic validates on startup
4. **Defaults** - sensible defaults for local development

In [ ]:
print("=== PollerConfig Fields ===")
print()
config = PollerConfig()
print(f"  scheduler:       {config.scheduler}")
print(f"  active_sports:   {config.active_sports}")
print(f"  log_level:       {config.log_level}")
print(f"  max_retries:     {config.max_retries}")
print(f"  retry_delay:     {config.retry_delay}s")
print(f"  prefect_api_url:  {config.prefect_api_url}")
print(f"  db.host:         {config.db.host}")
print(f"  db.port:         {config.db.port}")
print(f"  db.name:         {config.db.name}")
print(f"  odds_api.interval: {config.odds_api.interval_seconds}s")
print(f"  espn.interval:    {config.espn.interval_seconds}s")
print()
print("=== Environment Variable Mapping ===")
print("  QUANT_SPORTS_POLLER_SCHEDULER=prefect")
print("  QUANT_SPORTS_POLLER_ACTIVE_SPORTS=nfl,nba")
print("  QUANT_SPORTS_POLLER_LOG_LEVEL=INFO")
print("  QUANT_SPORTS_POLLER_MAX_RETRIES=3")
print("  QUANT_SPORTS_POLLER_RETRY_DELAY=30")
print("  QUANT_SPORTS_POLLER_DB__HOST=localhost")
print("  QUANT_SPORTS_POLLER_DB__PORT=5434")
print("  QUANT_SPORTS_POLLER_DB__NAME=quantitative_sports")
print("  QUANT_SPORTS_POLLER_ODDS_API__API_KEY=<secret>  # NEVER commit!")

---

## Section 14: Deployment Checklist

Before going live, verify every item on this checklist:

### Infrastructure
- [ ] TimescaleDB running and accessible
- [ ] Database migrations applied
- [ ] Connection pool configured (min/max connections)

### Configuration
- [ ] All environment variables set
- [ ] API keys in secrets (not in code)
- [ ] Active sports configured
- [ ] Scheduler selected (prefect or cron)

### Monitoring
- [ ] Prometheus metrics endpoint exposed
- [ ] Grafana dashboard imported
- [ ] Alert rules configured (pagerduty/slack/email)
- [ ] Health endpoints responding (/health, /ready)

### Data Quality
- [ ] Retention policies set
- [ ] Continuous aggregates created
- [ ] Backup schedule running

### Resilience
- [ ] Circuit breakers configured
- [ ] Retry with backoff enabled
- [ ] Graceful shutdown tested
- [ ] Database reconnection tested

### Logging
- [ ] Structured logging enabled
- [ ] Log levels set correctly (INFO for production)
- [ ] Correlation IDs flowing through requests
- [ ] Log aggregation configured (ELK/Loki)

In [ ]:
@dataclass
class ChecklistItem:
    """A deployment checklist item."""
    category: str
    item: str
    check: str
    critical: bool = True


DEPLOYMENT_CHECKLIST = [
    ChecklistItem("Infrastructure", "TimescaleDB running", "docker-compose ps", True),
    ChecklistItem("Infrastructure", "Migrations applied", "alembic upgrade head", True),
    ChecklistItem("Infrastructure", "Connection pool configured", "Check db.pool_size", True),
    ChecklistItem("Configuration", "Environment variables set", "env | grep QUANT_SPORTS", True),
    ChecklistItem("Configuration", "API keys in secrets", "Check .env or vault", True),
    ChecklistItem("Configuration", "Active sports configured", "Check ACTIVE_SPORTS", False),
    ChecklistItem("Configuration", "Scheduler selected", "Check SCHEDULER env var", True),
    ChecklistItem("Monitoring", "Prometheus metrics endpoint", "curl localhost:8000/metrics", True),
    ChecklistItem("Monitoring", "Grafana dashboard imported", "Check Grafana UI", False),
    ChecklistItem("Monitoring", "Alert rules configured", "Check alertmanager config", True),
    ChecklistItem("Monitoring", "Health endpoints responding", "curl localhost:8000/health", True),
    ChecklistItem("Data Quality", "Retention policies set", "Check TimescaleDB policies", False),
    ChecklistItem("Data Quality", "Continuous aggregates created", "Check materialized views", False),
    ChecklistItem("Data Quality", "Backup schedule running", "Check pg_dump cron", True),
    ChecklistItem("Resilience", "Circuit breakers configured", "Check PollerConfig.max_retries", True),
    ChecklistItem("Resilience", "Retry with backoff enabled", "Check retry settings", True),
    ChecklistItem("Resilience", "Graceful shutdown tested", "kill -SIGTERM <pid>", True),
    ChecklistItem("Resilience", "DB reconnection tested", "Stop/restart TimescaleDB", True),
    ChecklistItem("Logging", "Structured logging enabled", "Check log output format", False),
    ChecklistItem("Logging", "Log levels correct", "Check LOG_LEVEL env var", False),
    ChecklistItem("Logging", "Correlation IDs enabled", "Check log entries for correlation_id", False),
    ChecklistItem("Logging", "Log aggregation configured", "Check Loki/ELK", True),
]

print("Quant-Sports Deployment Checklist")
print("=" * 70)
current_category = ""
for item in DEPLOYMENT_CHECKLIST:
    if item.category != current_category:
        current_category = item.category
        print(f"\n  {item.category}:")
    priority = "CRITICAL" if item.critical else "RECOMMENDED"
    check_mark = "x" if item.critical else " "
    print(f"    [{check_mark}] [{priority}] {item.item}")
    print(f"        Verify: {item.check}")

critical_count = sum(1 for i in DEPLOYMENT_CHECKLIST if i.critical)
total_count = len(DEPLOYMENT_CHECKLIST)
print(f"\n{'='*70}")
print(f"Total items: {total_count} ({critical_count} critical, {total_count - critical_count} recommended)")

---

## Exercises

Try these on your own:

1. **Set up Prometheus** - Install `prometheus_client` and expose a metrics endpoint on port 8000. Configure Prometheus to scrape it every 15 seconds.

2. **Write a runbook for poller failures** - Document the steps to take when a poller is in `down` state (3+ consecutive failures). Include: how to diagnose, restart, and verify recovery.

3. **Implement a circuit breaker** - Extend the `CircuitBreaker` class to integrate with the poller health tracking. When the circuit opens, skip the poll cycle and log a warning.

4. **Create a Grafana dashboard** - Import the dashboard JSON from Section 9 into Grafana. Add panels for bankroll tracking, win rate, and P/L.

---

## Summary

In this lab you learned:

- **Scheduling** - Prefect vs cron vs Kubernetes CronJob, and when to use each
- **Error handling** - Retry with backoff, circuit breaker pattern, and per-task isolation
- **Data quality** - Schema checks, freshness validation, completeness audits
- **Alerting** - Querying `poller_health` and generating alerts on consecutive failures
- **Retention** - TimescaleDB `drop_chunks` and continuous aggregates for each table
- **Backup** - `pg_dump`, WAL archiving, and replication strategies
- **Monitoring** - Prometheus metrics and Grafana dashboards
- **Health endpoints** - `/health`, `/ready`, `/metrics` for load balancers and orchestration
- **Graceful shutdown** - SIGTERM handling, in-flight task completion, connection cleanup
- **Logging** - Structured JSON logging with correlation IDs
- **Configuration** - Environment variables, secrets, and `pydantic-settings` validation

### Key Configuration Reference

| Variable | Default | Purpose |
|---|---|---|
| `QUANT_SPORTS_POLLER_SCHEDULER` | `prefect` | Scheduler backend |
| `QUANT_SPORTS_POLLER_ACTIVE_SPORTS` | `nfl,nba` | Comma-separated sport list |
| `QUANT_SPORTS_POLLER_LOG_LEVEL` | `INFO` | Logging level |
| `QUANT_SPORTS_POLLER_MAX_RETRIES` | `3` | Max retry attempts |
| `QUANT_SPORTS_POLLER_RETRY_DELAY` | `30` | Seconds between retries |
| `QUANT_SPORTS_POLLER_DB__HOST` | `localhost` | Database host |
| `QUANT_SPORTS_POLLER_DB__PORT` | `5434` | Database port |
| `QUANT_SPORTS_POLLER_ODDS_API__API_KEY` | `` | Odds API key (secret!) |

### Next Steps

Congratulations on completing the Quant-Sports lab series! To go further:

- **Deploy**: Use Docker Compose to run the full stack
- **Automate**: Create Prefect flows for the full workflow
- **Scale**: Add more sports, books, and strategies
- **Monitor**: Build a production Grafana dashboard
- **Contribute**: Add your custom strategies back to the project

---

*This concludes the Quant-Sports Lab Series (Labs 01-10).*